# Domain 3: Claude Code Configuration & Workflows
## TS 3.2: Create and Configure Custom Slash Commands and Skills
**Certification Focus**: CCAR-F Domain 3 (20% Exam Weight) | **Scenario Alignment**: Scenario 2 & 4

---

### 🎯 Key Architectural Principles
1. **Slash Commands**: Declared in `.claude/commands/`. Expand arguments using `$ARGUMENTS` (full argument string), `$1`, `$2` (positional tokens).
2. **Claude Code Skills**: Declared in `.claude/skills/<name>/SKILL.md` with YAML frontmatter:
   - `name`: Unique skill identifier.
   - `description`: When Claude should trigger this skill.
   - `context: fork`: **Crucial architectural setting** that isolates the skill's execution into a separate sub-thread, preventing context bloat in the main session.
   - `allowed-tools`: Restricts which tools the skill may invoke.

In [ ]:
# Step 1: Environment & Setup
import re
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 3.2 Slash Commands & Skills.")

### ⚙️ Step 2: Skill & Command Parsers
We implement the parsing of custom slash commands and YAML frontmatter skills.

In [ ]:
# Step 2: Command & Skill Parsers
@dataclass
class SkillDefinition:
    name: str
    description: str
    context: str  # 'main' | 'fork'
    allowed_tools: List[str]
    body: str


class SkillCommandParser:
    @staticmethod
    def parse_slash_command(template: str, user_args: str) -> str:
        """Replaces $ARGUMENTS and $1, $2 with user-supplied arguments."""
        tokens = user_args.strip().split()
        expanded = template.replace("$ARGUMENTS", user_args.strip())
        for i, token in enumerate(tokens, start=1):
            expanded = expanded.replace(f"${i}", token)
        return expanded

    @staticmethod
    def parse_skill_markdown(skill_md: str) -> SkillDefinition:
        """Parses YAML frontmatter and markdown body from SKILL.md."""
        match = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)$", skill_md, re.DOTALL)
        if not match:
            raise ValueError("Invalid SKILL.md format: missing YAML frontmatter delimiters (---)")
        
        frontmatter_raw, body = match.groups()
        metadata = {}
        for line in frontmatter_raw.splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                k, v = k.strip(), v.strip()
                if v.startswith("[") and v.endswith("]"):
                    v = [item.strip() for item in v[1:-1].split(",") if item.strip()]
                metadata[k] = v

        return SkillDefinition(
            name=metadata.get("name", "unnamed"),
            description=metadata.get("description", ""),
            context=metadata.get("context", "main"),
            allowed_tools=metadata.get("allowed-tools", []),
            body=body.strip(),
        )


class ClaudeCodeRunner:
    def execute_skill(self, skill: SkillDefinition, target_path: str) -> Dict[str, Any]:
        """Executes skill, observing the context: fork isolation contract."""
        is_forked = (skill.context == "fork")
        return {
            "skill_name": skill.name,
            "execution_mode": "forked" if is_forked else "main_thread",
            "target_path": target_path,
            "allowed_tools": skill.allowed_tools,
            "main_context_bloat_prevented": is_forked,
        }

print(" Command & Skill parser engine initialized.")

### 🧪 Step 3: Interactive Simulation
1. Test slash command argument expansion (`/deploy branch stage`).
2. Parse a custom skill with `context: fork`.
3. Verify forked execution mode.

In [ ]:
# Step 3: Simulation
# 1. Slash Command Expansion
template = "git checkout $1 && git merge $ARGUMENTS && deploy --env=$2"
expanded = SkillCommandParser.parse_slash_command(template, "feature-auth staging")
print("Expanded Command:", expanded)

# 2. Skill Parsing
sample_skill_md = """---
name: security-audit
description: Audits target repository directory for security flaws.
context: fork
allowed-tools: [Read, Grep]
---
Inspect all files for hardcoded credentials and unsafe deserialization.
"""
skill = SkillCommandParser.parse_skill_markdown(sample_skill_md)
print(f"\nParsed Skill: {skill.name} (Context: {skill.context})")

# 3. Skill Execution
runner = ClaudeCodeRunner()
exec_res = runner.execute_skill(skill, "src/auth")
print("Skill Execution Result:", exec_res)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Positional tokens `$1`, `$2` and `$ARGUMENTS` are accurately expanded.
2. YAML frontmatter correctly extracts `context: fork` and `allowed-tools`.
3. Execution mode is `forked` and main context bloat is prevented.

In [ ]:
# Step 4: Verification Checks
assert "git checkout feature-auth" in expanded
assert "--env=staging" in expanded

assert skill.name == "security-audit"
assert skill.context == "fork"
assert "Read" in skill.allowed_tools
assert "Grep" in skill.allowed_tools

assert exec_res["execution_mode"] == "forked"
assert exec_res["main_context_bloat_prevented"] is True

print(" All verification assertions passed! Skills and commands verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Set `context: fork` in SKILL.md to execute exploratory/audit tasks without bloating main session context.
- **Rule 2**: Use `$ARGUMENTS` in slash command templates for the entire query string, and `$1`, `$2` for positional tokens.
- **Rule 3**: Restrict `allowed-tools` in skill frontmatter to enforce least privilege.